# Calculations and demos of the perturbation theory

- Bernardeau et al (https://arxiv.org/pdf/astro-ph/0112551)

In [ ]:
import toml
import numpy as np
from scipy.interpolate import CubicSpline, RegularGridInterpolator

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

import h5py
import camb
from colossus.cosmology import cosmology

In [ ]:
import logging

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
# Facecolor values from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#ffffff',
    # 'figure.facecolor': '#f4f0e8',
    'axes.facecolor': '#ffffff',
    # 'axes.facecolor': '#f4f0e8',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)
plt.rcParams['text.usetex'] = False

### Auxiliary functions

In [ ]:
class RNG:
    def __init__(self, seed=None):
        self.rng = np.random.default_rng(seed)
    def _get_rng(self, seed=None):
        '''
        Returns a new generator if seed is provided, otherwise returns
        the existing one.
        '''
        return np.random.default_rng(seed) if seed is not None else self.rng
    def uniform(self, size=None, seed=None):
        '''Generate uniformly distributed random numbers.'''
        return self._get_rng(seed).uniform(size=size)
    def normal(self, mean=0.0, std=1.0, size=None, seed=None):
        '''Generate normally distributed random numbers.'''
        return self._get_rng(seed).normal(loc=mean, scale=std, size=size)
    def integers(self, low, high=None, size=None, seed=None):
        '''Generate random integers.'''
        return self._get_rng(seed).integers(low, high=high, size=size)

In [ ]:
seed = 137
rng = RNG(seed=seed)

In [ ]:
def interpolate_field(x, field, dk, method='linear'):
    r'''
    Interpolate a grid-based field onto particle positions using periodic
    boundaries.

    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Particle positions in the simulation box.
    field : ndarray
        The grid-based field (e.g. a displacement field) defined on a
        regular grid.
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    method : str
        The interpolation method to use. This can be 'linear', 'nearest',
        or 'cubic'. The default is 'linear'.

    Returns
    -------
    interp_values : ndarray of shape (N,)
        Field values interpolated at the particle positions.
    '''
    nvox = field.shape
    mesh = tuple(np.arange(-(n-1)*dk/2, n*dk/2, dk) for n in nvox)
    interpolator = RegularGridInterpolator(
        points=mesh,
        values=field,
        method=method,
        bounds_error=False,
        fill_value=None  # Extrapolate using periodic wrapping if needed
    )
    return interpolator(x)

In [ ]:
def create_grid(nvox, dk):
    '''
    Create a regular grid for the simulation box.

    Parameters
    ----------
    nvox : tuple of int
        The number of voxels in each dimension of the simulation box.
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.

    Returns
    -------
    particles : ndarray of shape (N, 3)
        The particle positions in the simulation box, where N is the total
        number of particles (voxels).
    coords : ndarray of shape (3, Nx, Ny, Nz)
        The grid coordinates in each dimension, where Nx, Ny, Nz are the
        number of voxels in each dimension.
    '''
    mesh = [np.arange(-(n-1)*dk/2, n*dk/2, dk) for n in nvox]
    xx, yy, zz = np.meshgrid(*mesh, indexing='ij')
    particles = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=1)
    return particles, np.array((xx, yy, zz))

In [ ]:
def create_particles(npart: int, Lbox, seed=None):
    r'''
    Create a set of particles uniformly distributed in a cubic box.

    Parameters
    ----------
    npart : int
        The number of particles to create.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz] or a single
        float value for a cubic box.
    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    particles : ndarray of shape (npart, 3)
        Particle positions in the simulation box.
    '''
    return rng.uniform(size=(npart, 3), seed=seed) * np.array(Lbox)

In [ ]:
def compute_overdensity(density_field):
    r'''
    Compute the overdensity field defined as
    .. math::
        $\delta(\mathbf{x}) = \frac{\rho(\mathbf{x}) - \bar{\rho}}{\bar{\rho}}$.

    where :math:`\rho(\mathbf{x})` is the density field and :math:`\bar{\rho}`
    is the mean density.

    Parameters
    ----------
    density_field : ndarray
        The density field.

    Returns
    -------
    overdensity_field : ndarray
        The overdensity field :math:`\delta(\mathbf{x})`.
    '''
    mean_density = np.mean(density_field)
    return (density_field - mean_density) / mean_density

### Cosmology

In [ ]:
def load_cosmo_parameters(
        path, parameter_set='Planck2018EE+BAO+SN', value_type='best'):
    '''TODO'''
    if value_type not in ['best', 'mean']:
        raise ValueError("The 'value_type' must be either 'best' or 'mean'!")

    with open(path, 'r') as f:
        config_data = toml.load(f)

    if parameter_set not in config_data:
        raise KeyError(f"Parameter set '{parameter_set}' not found in the TOML data.")

    config_data = config_data[parameter_set]

    params = {}
    for key, value in config_data.items():
        if isinstance(value, dict):
            # Values with best fit/68% limit values
            params[key] = value[value_type]
            if value_type == 'mean':
                params[f'{key}_err'] = value['error']
        else:
            # Single value parameters
            params[key] = value

    return params

In [ ]:
def calculate_cosmo_parameters(params):
    '''
    Calculate cosmological parameters from the loaded configuration.
    '''
    h = params['H0'] / 100.0
    omega_nu = params.get('MNU', 0.06) / 93.14 / h**2
    omega_c = params['OMEGA_M'] - params['OMEGA_B'] - omega_nu
    params.update({'H': h, 'OMBH2': params['OMEGA_B']*h**2, 'OMCH2': omega_c*h**2})
    return params

In [ ]:
params = load_cosmo_parameters('../stepsic/config/cosmology.toml')
params = calculate_cosmo_parameters(params)
params

In [ ]:
def hubble_a(a, H0, omega_m, omega_l):
    r'''
    Computes the Hubble parameter :math:`H(a)` at scale factor :math:`a`.

    The Hubble parameter is given by

    .. math::
        H(a) = H_0\,\sqrt{\Omega_m\,a^3 + (1 - \Omega_m - \Omega_\Lambda)\,a^2 + \Omega_\Lambda},

    where :math:`a` is the scale factor normalized to 1 at present. :math:`H_0`
    is the Hubble constant, :math:`\Omega_m` is the present-day matter
    density parameter and :math:`\Omega_\Lambda` is the present-day dark
    energy density parameter.

    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    H0 : float
        Hubble constant in km/s/Mpc.
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The Hubble parameter :math:`H(a)`.
    '''
    return H0 * np.sqrt(omega_m / a**3 + (1 - omega_m - omega_l) / a**2 + omega_l)

In [ ]:
def F_omega(a, omega_m, omega_l):
    r'''
    Computes the linear growth rate factor for first-order Lagrangian
    perturbation.

    This function returns the factor :math:`F_\omega(a)`, defined by

    .. math::
        F_\omega(a)  = \frac{d\ln(D_1)}{d\ln(a)},

    where the effective matter density parameter :math:`\omega(a)` is computed
    as

    .. math::
        \Omega(a) = \frac{\omega_m}{\omega_m + a\,(1 - \omega_m - \omega_l) + \omega_l\,a^3}.

    :math:`F_\omega` approximates the logarithmic derivative of the linear
    growth factor :math:`D_1` with respect to the scale factor :math:`a`, i.e.

    .. math::
        f \equiv \frac{d\ln(D_1)}{d\ln(a)}.


    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The linear growth rate :math:`F_\omega(a)`.
    '''
    omega_a = omega_m / (omega_m + a * (1 - omega_m - omega_l) + a**3 * omega_l)
    return np.power(omega_a, 5.0/9.0)  # Bernardeau et al. 2001, eq. 101a

In [ ]:
def F2_omega(a, omega_m, omega_l):
    r'''
    Computes the second-order growth rate factor for second-order
    Lagrangian perturbation theory corrections.

    This function returns the factor :math:`F2_\omega(a)`, defined by

    .. math::
        F2_\omega(a) = \frac{d\ln(D_2)}{d\ln(a)},

    where the effective matter density parameter :math:`\omega(a)` is computed
    as

    .. math::
        \omega(a) = \frac{\omega_m}{\omega_m + a\,(1 - \omega_m - \omega_l) + \omega_l\,a^3}.

    :math:`F2_\omega` is used in second-order Lagrangian perturbation theory
    to scale the second-order displacement field and its time derivative,
    thereby accounting for non-linear corrections to the growth of structure.

    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The second-order growth rate :math:`F2_\omega(a)`.
    '''
    omega_a = omega_m / (omega_m + a * (1 - omega_m - omega_l) + a**3 * omega_l)
    return 2 * np.power(omega_a, 6.0/11.0)  # Bernardeau et al. 2001, eq. 101b

In [ ]:
class ColossusCosmology:
    def __init__(self, *,
            H0=67.742, Om0=0.3099, Ob0=0.048891, Ol0=0.6901, sigma8=0.8105,
            ns=0.96822, Neff=3.046, w0=-1.0, wa=0.0, **kwargs):
        '''
        Wrapper to initialize a Colossus cosmology object.

        Parameters
        ----------
        H0 : float
            Hubble constant in km/s/Mpc.
        Om0 : float
            Total matter density parameter today divided by the critical density.
        Ob0 : float
            Baryon density parameter today divided by the critical density.
        Ol0 : float
            Dark energy density parameter today divided by the critical density.
        sigma8 : float
            RMS matter fluctuation amplitude at redshift 0.
        ns : float
            Scalar spectrum power-law index for k_pivot = 0.05 Mpc^-1.
        Neff : float
            Total effective number of massive and massless neutrinos.
        w0 : float
            Dark energy equation of state parameter at redshift 0.
        wa : float
            Dark energy equation of state parameter evolution.
        '''
        flat = np.isclose(1 - Ol0 - Om0, 0.0, rtol=1e-5, atol=1e-8)
        de_model = 'w0wa' if wa != 0.0 else 'w0' if w0 != -1.0 else 'lambda'
        self.cosmo = cosmology.setCosmology(
            'steps', H0=H0, Om0=Om0, Ob0=Ob0, sigma8=sigma8, ns=ns, Neff=Neff,
            w0=w0, wa=wa, flat=flat, de_model=de_model, **kwargs)
    
    def Dzplus0(self, z : float) -> float:
        '''
        Calculate the linear growth factor :math:`D_+(z)` normalized to
        1 at present.

        For :math:`w_0` and CPL dark energy, we use the Colossus
        implementation of Eq. (11) from Linder & Jenkins (2003).
        See paper at https://arxiv.org/pdf/astro-ph/0305286.pdf.
        
        Parameters
        ----------
        z : float
            Redshift(s) at which to compute the growth factor.
        
        Returns
        -------
        float or np.ndarray
            The linear growth factor D_+(z).
        '''
        Dzplus = self.cosmo.growthFactorUnnormalized(z)
        D0plus = self.cosmo.growthFactorUnnormalized(0.0)

        log.info(f'D_+({z=:.2f})/D_+(z=0) = {Dzplus / D0plus:.2e}')
        return Dzplus / D0plus

### Power spectrum

In [ ]:
class CAMBCosmology:
    def __init__(self, *,
            H0=67.742, ombh2=0.022436, omch2=0.11914, omk=0.0, mnu=0.06,
            nnu=3.046, YHe=0.245421, zrei=7.89, TCMB=2.775, w0=-1.0, wa=0.0,
            nonlinear=False, halofit_version='mead2020', **kwargs):
        '''
        Wrapper to initialize a CAMB cosmology object.
        
        Parameters
        ----------
        H0 : float
            Hubble constant in km/s/Mpc.
        ombh2 : float
            Baryon density parameter today.
        omch2 : float
            Cold dark matter density parameter today.
        omk : float
            Curvature density today divided by the critical density.
        mnu : float
            Sum of active neutrino masses in eV.
        nnu : float
            Total effective number of massive and massless neutrinos.
        YHe : float
            Fraction of baryonic mass in helium. Set to `None` to be
            calculated internally for BBN consistency.
        TCMB : float
            CMB temperature in Kelvin.
        zrei : float
            Redshift at which the Universe is half reionized.
        w0 : float
            Dark energy equation of state parameter at redshift 0.
        wa : float
            Dark energy equation of state parameter evolution.
        nonlinear : bool
            If True, include non-linear corrections using Halofit.
        halofit_version : str
            Version of the Halofit model to use for non-linear corrections.
            Check ``camb.nonlinear.Halofit`` for available models.
        '''
        self.params = camb.CAMBparams()
        self.params.set_cosmology(
            H0=H0, ombh2=ombh2, omch2=omch2, omk=omk, mnu=mnu, nnu=nnu,
            YHe=YHe, TCMB=TCMB, zrei=zrei, **kwargs)
        if w0 != -1.0 or wa != 0.0:
            log.info(f'Using single fluid dark energy model with w0={w0} and wa={wa}')
            self.params.DarkEnergy = camb.dark_energy.DarkEnergyFluid()
            self.params.DarkEnergy.set_params(w=w0, wa=wa)

        if nonlinear:
            log.info(f'Using non-linear corrections with Halofit model `{halofit_version}`')
            self.params.NonLinear = camb.model.NonLinear_both
            self.params.NonLinearModel = camb.nonlinear.Halofit()
            self.params.NonLinearModel.set_params(halofit_version=halofit_version)
        else:
            log.info('Using linear theory only.')
            self.params.NonLinear = camb.model.NonLinear_none

    def get_sigma8(self, z=0, *, As=2.1064e-09, ns=0.96822, kmax=1.0):
        r'''
        Calculate the RMS matter fluctuation amplitude :math:`\sigma_8`
        at given redshift ``z`` using CAMB.
        
        Parameters
        ----------
        z : float
            Target redshift.
        As : float
            Comoving curvature power at :math:`k = 0.05\,\mathrm{Mpc}^{-1}`.
            This is the amplitude of the primordial power spectrum at large
            scales, typically set to match the observed :math:`\sigma_8`.
        ns : float
            Scalar spectral index.
        kmax : float
            Maximum wavenumber in :math:`h^{-1}\,\mathrm{Mpc}`.

        Returns
        -------
        float
            The RMS matter fluctuation amplitude $\sigma_8$ at redshift $z$.
        '''
        self.params.InitPower.set_params(As=As, ns=ns)
        self.params.set_matter_power(redshifts=[z], kmax=kmax)
        results = camb.get_results(self.params)
        sigma8 = results.get_sigma8()[0]
        log.info(f'RMS matter fluctuation amplitude {sigma8 = :.4f} (from {As = :.3e})')
        return sigma8
    
    def get_spectrum(self, *,
            z=127, As=2.1064e-09, ns=0.96822, sigma8_init=None,
            kmin=0.01, kmax=1.0, npoints=512):
        r'''
        Calculate the matter power spectrum using CAMB.

        Parameters:
        -----------
        z : float or list of float
            Redshifts at which the linear power spectrum is calculated.
        As : float
            Comoving curvature power at :math:`k = 0.05\,\mathrm{Mpc}^{-1}`.
            This is the amplitude of the primordial power spectrum at large
            scales, typically set to match the observed :math:`\sigma_8`.
        ns : float
            Scalar spectral index.
        sigma8_init : float
            Rescale the matter power spectrum to the given :math:`\sigma_8`
            value.
        kmin : float
            Minimum wavenumber in :math:`h^{-1}\,\mathrm{Mpc}`.
        kmax : float
            Maximum wavenumber in :math:`h^{-1}\,\mathrm{Mpc}`.
        npoints : int
            Number of wavenumber points.
        '''
        # Optional rescaling of the `As` amplitude to match a desired sigma8
        if sigma8_init is not None:
            sigma8 = self.get_sigma8(z=0, As=As, ns=ns, kmax=kmax)
            As *= (sigma8_init / sigma8)**2
            log.info(f'Rescaling matter fluctuation amplitude...')
            sigma8 = self.get_sigma8(z=0, As=As, ns=ns, kmax=kmax)

        # Calculating P(k) at redshift `z`
        self.params.set_matter_power(redshifts=np.atleast_1d(z).tolist(), kmax=kmax)
        results = camb.get_results(self.params)
        kh, _, pk = results.get_matter_power_spectrum(
                                    minkh=kmin, maxkh=kmax, npoints=npoints)
        pk3 = pk * kh**3/(2*np.pi**2)  # Save (log(kh), log(pk3)).T for StePS/Gadget
        return kh, pk, pk3

## Fourier grid

In the context of cosmological simulations, the majority of calculations are performed in Fourier space for both numerical efficiency and convenience.

The `fourier_grid` function constructs a three-dimensional Fourier space grid, which serves as the main ... cosmological simulations. This grid represents the wavevector components and their magnitudes in Fourier space, enabling operations such as filtering, convolution, and spectral analysis.

In [ ]:
def cubic_voxels(nmesh, Lbox):
    '''
    Defines a rectangular cuboid mesh with the specified number of
    voxels in each dimensions, ensuring that the voxels are cubic.
    The function calculates the number of voxels in each dimension
    (Nx, Ny, Nz) based on the shortest dimension of the cuboid and
    scales the other dimensions accordingly.

    Parameters
    ----------
    nmesh : int
        Number of voxels in the shortest dimension.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz] or a single
        float value for a cubic box.

    Returns
    -------
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    '''
    Lbox = np.broadcast_to(Lbox, (3,))
    nvox = np.ceil(Lbox / (np.min(Lbox) / nmesh)).astype(int)
    nvox = (nvox + nvox % 2).astype(int)  # Ensure even number of voxels
    dk = np.min(Lbox) / np.min(nvox)
    log.info('mesh: Nx={}, Ny={}, Nz={}; step size: {}'.format(*nvox, dk))
    return nvox, dk

In [ ]:
def fourier_grid(nvox, dk, hermitian=False):
    r'''
    Construct a 3D Fourier space grid.

    This function generates a three-dimensional array of wavevector
    components (``kvec``) and computes the corresponding magnitude
    (``kmod``) for a cubic grid with ``nmesh`` points per side within
    a box of size ``Lbox``.

    The grid is then constructed using the FFT frequencies:
    - For the first two dimensions, the full set of FFT frequencies is
      computed using ``np.fft.fftfreq``.
    - For the third dimension, if the input field is real-valued (i.e.
      if Hermitian symmetry is assumed), the reduced set of frequencies
      is computed using ``np.fft.rfftfreq``.

    Parameters
    ----------
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    hermitian : bool
        If True, assume the field has Hermitian symmetry (i.e., it is
        real-valued) and use the reduced FFT along the last dimension.

    Returns
    -------
    kvec : ndarray
        A three-dimensional array of wavevector components with shape:
          - :math:`(3, {N_x}, {N_y}, {N_z}//2+1)` if ``hermitian`` is True.
          - :math:`(3, {N_x}, {N_y}, {N_z})` if ``hermitian`` is False.
        Each sub-array corresponds to the ``x``, ``y``, or ``z`` component
        of the wavevector.
    
    kmod : ndarray
        The magnitude of the wavevector at each grid point, computed as
        :math:`\|\mathbf{k}\| = \sqrt{k_x^2 + k_y^2 + k_z^2}`.
    '''
    kx = np.fft.fftfreq(nvox[0]) * 2 * np.pi / dk
    ky = np.fft.fftfreq(nvox[1]) * 2 * np.pi / dk
    if hermitian:
        kz = np.fft.rfftfreq(nvox[2]) * 2 * np.pi / dk
    else:
        kz = np.fft.fftfreq(nvox[2]) * 2 * np.pi / dk
    kvec = np.array(np.meshgrid(kx, ky, kz, indexing='ij'))
    kmod = np.linalg.norm(kvec, axis=0)
    return kvec, kmod

### Sample physical overdensity from the power spectrum 

In [ ]:
def white_noise(nvox, seed=None):
    r'''
    Return a complex Gaussian array :math:`W(k)` on the ``rfftn()`` grid
    `(Nx, Ny, Nz//2+1)`, obeying Hermitian constraints that guarantee
    :math:`\delta(x)` reconstructed with ``irfftn()`` is real.

    The field is generated in the real space.

    Parameters
    ----------
    nvox : tuple of int
        Number of voxels in each dimension (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    seed : int or None, optional
        Random seed for reproducibility. If None, uses the default RNG.

    Returns
    -------
    w_k : ndarray
        3D array of white noise values.
    '''
    w_k = np.fft.rfftn(rng.normal(size=nvox, seed=seed))
    w_k[0, 0, 0] = 0.0  # set DC=0 (mean density) as we only need fluctuations
    return w_k

In [ ]:
def generate_delta_k(kh, pk, nvox, dk, *, field=None, seed=None):
    r'''
    Generates the Fourier modes of an arbitrary input field from a
    given power spectrum.

    Parameters
    ----------
    kh : ndarray
        1D array of wavenumbers (k), in h/Mpc.
    pk : ndarray
        1D array of the matter power spectrum P(k) at the initial redshift.
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    field : ndarray
        Fourier transform of the real-valued overdensity field
        :math:`\delta(\mathbf{x})` defined on a regular grid, where
        :math:`\mathbf{x}` are the comoving coordinates.
    seed : int
        The seed for the random number generator.

    Returns
    -------
    delta_k : ndarray
        A 3D complex-valued array of shape (Nx, Ny, Nz//2+1) representing
        the Fourier modes of the overdensity field.
    '''
    _, kmod = fourier_grid(nvox, dk, hermitian=True)
    
    # interpolate the power spectrum in log-log space
    spline = CubicSpline(np.log(kh), np.log(pk), extrapolate=True)
    pk_grid = np.zeros_like(kmod, dtype=float)
    mask = kmod > 0
    if np.any(mask):
        ktarget_log = np.log(kmod[mask])
        pk_grid[mask] = np.exp(spline(ktarget_log))

    if field is None:
        field = white_noise(size=nvox, seed=seed)

    # Sirko 2005; Bagla & Padmanabhan 1997; Klypin & Holtzman 1997
    return field * np.sqrt(pk_grid / dk**3)

In [ ]:
def log_lpt(x, xpert, vpert, *, title=None) -> None:
    '''TODO'''
    xabs, vabs = np.abs(xpert - x), np.abs(vpert)
    xmax, xavg = np.max(xabs, axis=0), np.mean(xabs, axis=0)
    vmax, vavg = np.max(vabs, axis=0), np.mean(vabs, axis=0)
    for i, xi in enumerate(('x', 'y', 'z')):
        log.info(f"{title} '{xi}' displacements:\t"
                 f"d_max = {xmax[i]*1e3:.3f} kpc/h; d_avg = {xavg[i]*1e3:.3f} kpc/h")
        log.info(f"{title} 'v{xi}' velocities:\t"
                 f"v_max = {vmax[i]:.3f} km/s; v_avg = {vavg[i]:.3f} km/s")
    return

## 1LPT - Zel'dovich approximation

In [ ]:
def lpt1(x, delta_k, nvox, dk, aHf1, h, counter=False):
    r'''
    Apply first-order Lagrangian Perturbation Theory (LPT), i.e., the
    Zel'dovich approximation, to generate perturbed particle positions
    and velocities.

    In this approximation, particles are displaced from their initial
    (Lagrangian) positions :math:`\mathbf{q}` to their final (Eulerian)
    positions :math:`\mathbf{x}` using a displacement field
    :math:`\mathbf{\Psi}`:

    .. math::
        \mathbf{x}(\mathbf{q}, t) = \mathbf{q} + \mathbf{\Psi}(\mathbf{q}),

    where :math:`D_1(t)` is the linear growth factor and :math:`\mathbf{\Psi}(\mathbf{q})`
    is the displacement field computed from the initial density
    perturbations. The displacement field is related to the gravitational
    potential, and in Fourier space, it is calculated from the
    overdensity field :math:`\delta(\mathbf{k})`:

    .. math::
        \mathbf{\Psi}(\mathbf{k}) =
            -i \, \frac{\mathbf{k}}{|\mathbf{k}|^2} \, \delta(\mathbf{k})
        \quad \text{for } |\mathbf{k}| > 0\,,

    where :math:`\mathbf{k}` is the wavevector.
    
    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Initial unperturbed particle positions (Lagrangian coordinates),
        in comoving Mpc/h.
    delta_k : ndarray
        A 3D complex-valued array of shape (Nx, Ny, Nz//2+1) representing
        the Fourier modes of the overdensity field.
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    D1 : float
        The linear growth factor :math:`D_1(z)` at the desired redshift.
    aHf1 : float
        A prefactor for the velocity calculation, typically related to the
        time derivative of the growth factor (e.g. :math:`\dot{D}_1` or
        :math:`H(a)f(a)` where f is the growth rate). The code uses this
        in a non-standard velocity formula.
    h : float
        The dimensionless Hubble parameter, :math:`h = H_0 / 100`, where
        :math:`H_0` is the Hubble constant.
    counter : bool
        If True, applies a global sign flip to the Fourier-space density
        field (equivalent to a :math:`\pi` phase shift). This is useful
        for running "counter-phased" simulations to reduce sample
        variance. See more in Angulo-Pontzen (2017).
    
    Returns
    -------
    xpert : ndarray of shape (N, 3)
        Perturbed particle positions (Eulerian coordinates) in comoving
        Mpc/h. Positions are wrapped to lie within the periodic box.
    vpert : ndarray of shape (N, 3)
        Particle peculiar velocities in km/s. The velocity is computed
        according to the specific formula implemented in this function:

        .. math::
            \mathbf{v} = \frac{\dot{D}(a)}{D(1)} \cdot \mathbf{\Psi} / h\,,
        
        where :math:`\dot{D}(a)` is approximated by the velocity prefactor

        .. math::
            v_{\mathrm{fac}} = a H(a) f(a)\,.

    Notes
    -----
    **Overview of the Implementation:**

    1.  **Fourier Grid Setup:** A 3D Fourier grid (``kvec``) is constructed
        for a mesh of size ``(Nx, Ny, Nz)``. For a real-valued field,
        according to the Hermitian constraints, a reduced FFT is used,
        so the k-space arrays have a shape of ``(Nx, Ny, Nz//2 + 1)``.
        Regardless of the mesh size, we always cut along the z-axis in
        this implementation.

    2.  **Displacement Field Calculation:**
        - The gravitational potential :math:`\phi(\mathbf{k})` is computed
          in Fourier space from the overdensity field :math:`\delta(\mathbf{k})`
          using the relation:
          .. math::
              \phi(\mathbf{k}) = -\frac{\delta(\mathbf{k})}{|\mathbf{k}|^2}\,,
          where :math:`|\mathbf{k}|` is the magnitude of the wavevector.
        - The Fourier-space displacement field :math:`\mathbf{\Psi}(\mathbf{k})`
          is computed for each spatial component. Division by zero at the
          DC mode (:math:`|\mathbf{k}| = 0`) is avoided.
        - An inverse FFT converts :math:`\mathbf{\Psi}(\mathbf{k})` back
          to a real-space grid.

    3.  **Interpolation and Particle Update:**
        - The gridded displacement field :math:`\mathbf{\Psi}` is
          interpolated to the Lagrangian particle positions :math:`\mathbf{q}`.
        - Particle positions are updated to their Eulerian coordinates:

          .. math::
              \mathbf{x}_{\text{pert}} = \mathbf{q} + \mathbf{\Psi}(\mathbf{q})

        - Particle velocities are computed using the interpolated
          displacement field :math:`\mathbf{\Psi}`, the growth factor
          ``D1``, and the velocity prefactor ``aHf1``. The final result
          is multiplied by ``h`` to obtain units of km/s.
    '''
    kvec, kmod = fourier_grid(nvox, dk, hermitian=True)
    delta_k = delta_k * np.exp(1j * np.pi) if counter else delta_k
    mask = kmod > 0.0  # Avoid division by zero at k = 0
    phi_k = np.zeros_like(kmod, dtype=complex)
    phi_k[mask] = -delta_k[mask] / kmod[mask]**2  # Gravitational potential in Fourier space
    psi1_k = -1j * phi_k[np.newaxis, ...] * kvec  # Displacement field in Fourier space
    disp_field = np.fft.irfftn(psi1_k, s=nvox, axes=(-3, -2, -1))
    disp_field_interp = np.empty_like(x, dtype=np.float32)
    for i in range(3):
        disp_field_interp[:, i] = interpolate_field(x, disp_field[i], dk)
    xpert = x + disp_field_interp  # Bernardeau et al. 2001, eq. 98
    vpert = disp_field_interp * aHf1  # Bernardeau et al. 2001, eq. 99
    return xpert, vpert / h # Mpc/h, km/s

In [ ]:
z = 63
a = 1.0 / (1.0 + z)

cosmo_colossus = ColossusCosmology(
    H0=params['H0'], Om0=params['OMEGA_M'], Ob0=params['OMEGA_B'],
    Ol0=params['OMEGA_L'], sigma8=params['SIGMA8'], ns=params['NS'],
    Neff=params['NNU'], w0=params['W0'], wa=params['WA'], Tcmb0=1e-6)
g1 = 1
D1 = g1 * cosmo_colossus.Dzplus0(z)
g2 = - 3.0/7.0 * params['OMEGA_M']**(-1/143)
D2 = g2 * D1**2  # Bernardeau et al. 2001, eq. 97  # Unused!
print(f'D1(z={z}) = {D1:.6f}, D2(z={z}) = {D2:.6f}')

# Bernardeau et al. 2001, eq. 99
# velocity prefactors should be in km/s/(Mpc/h)
aHf1 = a * hubble_a(a, params['H0'], params['OMEGA_M'], params['OMEGA_L'])
aHf1 *= F_omega(a, params['OMEGA_M'], params['OMEGA_L'])
aHf2 = a * hubble_a(a, params['H0'], params['OMEGA_M'], params['OMEGA_L'])
aHf2 *= F2_omega(a, params['OMEGA_M'], params['OMEGA_L'])
print(f'1st vel. prefac(z={z}) = {aHf1:.6f}, 2nd vel. prefac(z={z}) = {aHf2:.6f}')

In [ ]:
nmesh = 128
Lbox = 200
periodic = [0, 0, 0]
nvox, dk = cubic_voxels(nmesh, Lbox)

In [ ]:
kmin = np.pi / np.min(Lbox)
kmax = 100.0
npoints = 2048
cosmo_camb = CAMBCosmology(
    H0=params['H0'], ombh2=params['OMBH2'], omch2=params['OMCH2'],
    omk=params.get('OMEGA_K', 0.0), mnu=params['MNU'], nnu=params['NNU'],
    YHe=params['YHE'], TCMB=params['TCMB'], zrei=params['ZREI'],
    w0=params['W0'], wa=params['WA'], nonlinear=False)
kh, pk, pk3 = cosmo_camb.get_spectrum(
    z=0, As=params['AS'], ns=params['NS'], sigma8_init=params['SIGMA8'],
    kmin=kmin, kmax=kmax, npoints=npoints)
pk = pk[0]*D1**2  # Backscale P(k,z=0) with D1^2 to desired `z`

In [ ]:
field = white_noise(nvox=nvox, seed=seed)
# white noise field for MONOPHONIC
with h5py.File('output/initial_conditions.hdf5', 'w') as f:
    f.create_dataset('ic_white_noise', data=np.fft.irfftn(field))
delta_k = generate_delta_k(kh, pk, nvox, dk, field=field)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

if not isinstance(Lbox, (list, tuple, np.ndarray)):
    Lbox = (Lbox,) * 3
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis
grid = np.meshgrid(*(np.arange(-(n-1)*dk/2, n*dk/2, dk) for n in nvox), indexing='ij')
delta_x = np.fft.irfftn(delta_k)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, deltax_s = x1[slicer], x2[slicer], delta_x[slicer]

    # im = ax.pcolormesh(x1_s, x2_s, deltax_s, shading='auto')
    im = ax.imshow(deltax_s.T)
    # ax.set_xlim(-Lbox[idx[0]]/2, Lbox[idx[0]]/2)
    # ax.set_ylim(-Lbox[idx[1]]/2, Lbox[idx[1]]/2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]', fontsize=10)
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]', fontsize=10)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)
    cb = fig.colorbar(im, orientation='horizontal', cax=cax)
    cax.xaxis.set_ticks_position('top')
    cax.xaxis.set_label_position('top')
    cb.set_label(r'Overdensity $\delta(x)$', fontsize=10)
plt.show()

### Generate particles

In [ ]:
# x = create_particles(npart=32**3, Lbox=Lbox, seed=seed)
x, _ = create_grid(nvox, dk)
with h5py.File(f'./output/glass{nmesh}.hdf5', 'w') as f:
    header = f.create_group('Header')
    if isinstance(Lbox, (list, tuple, np.ndarray)):
        header.attrs['BoxSize'] = float(Lbox[0])
    else:
        header.attrs['BoxSize'] = float(Lbox)
    
    g = f.create_group('PartType1')
    g.create_dataset('Coordinates', data=x, dtype='f8')

In [ ]:
def inspect_glass_file(filename):
    with h5py.File(filename, 'r') as f:
        print('File structure:')
        def print_structure(name, obj):
            print(f'  {name}: {type(obj).__name__}')
            if hasattr(obj, 'shape'):
                print(f'    Shape: {obj.shape}')
            if hasattr(obj, 'dtype'):
                print(f'    Dtype: {obj.dtype}')
        
        f.visititems(print_structure)
        
        # Check specific attributes
        if 'Header' in f and 'BoxSize' in f['Header'].attrs:
            print(f"BoxSize: {f['Header'].attrs['BoxSize']}")
        
        if 'PartType1/Coordinates' in f:
            coords = f['PartType1/Coordinates'][:]
            print(f'Coordinates shape: {coords.shape}')
            print(f'First few particles:\n{coords[:3]}')

In [ ]:
inspect_glass_file(f'./output/glass{nmesh}.hdf5')

### Load glass

In [ ]:
def load_input_glass(filepath, Lbox, periodic):
    with h5py.File(filepath, 'r') as hdf:
        # particleIDs = hdf[f'/PartType1/ParticleIDs'][:]
        x = hdf[f'/PartType1/Coordinates'][:]
        # velocities = hdf[f'/PartType1/Velocities'][:]
        # masses = hdf[f'/PartType1/Masses'][:]
    mask = (x.max(axis=0) < np.multiply(Lbox, 0.5)) | periodic
    return np.where(mask, x, x - np.multiply(Lbox, 1.0)/2)

In [ ]:
GLASSFILE = f'../cylindrical-ic/Glass_Cylindrical_Lz100_R500_DS75_N48k.hdf5'
x = load_input_glass(GLASSFILE, Lbox=Lbox, periodic=periodic)
inspect_glass_file(GLASSFILE)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

if not isinstance(Lbox, (list, tuple, np.ndarray)):
    Lbox = (Lbox,) * 3

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]

    ax.scatter(x.T[idx[0]], x.T[idx[1]], s=1**2, c='k', alpha=0.5)
    # ax.set_xlim(-Lbox[idx[0]]/2, Lbox[idx[0]]/2)
    # ax.set_ylim(-Lbox[idx[1]]/2, Lbox[idx[1]]/2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]', fontsize=10)
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]', fontsize=10)
plt.show()

In [ ]:
xpert, vpert = lpt1(x, delta_k, nvox, dk, aHf1, h=params['H'], counter=False)
log_lpt(x, xpert, vpert, title='1-LPT')
# xpert = np.where(periodic, np.mod(xpert, Lbox), xpert)

In [ ]:
if x.size <= 3*36**3:
    nr, nc = 1, 2
    fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

    idx = [0, 1]

    ax = axes[0]
    ax.scatter(*x[:, idx].T, c='k', s=1**2, ec='none', alpha=0.5)
    ax.set_title('Initial positions', loc='left', fontsize=10)

    ax = axes[1]
    ax.scatter(*xpert[:, idx].T, c='tab:red', s=1**2, ec='none', alpha=0.5)
    ax.set_title('Perturbed (LPT1) positions', loc='left', fontsize=10)

    labels = ['x', 'y', 'z']
    for i, ax in enumerate(axes.flat):
        ax.set_aspect(1)
        #ax.set_xlim(0, Lbox[idx[0]])
        #ax.set_ylim(0, Lbox[idx[1]])
        ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
        ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    plt.show()

In [ ]:
if x.size <= 3*36**3:
    nr, nc = 1, 2
    fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

    idx = [1, 2]

    ax = axes[0]
    ax.scatter(*x[:, idx].T, c='k', s=1**2, ec='none', alpha=0.5)
    ax.set_title('Initial positions', loc='left', fontsize=10)

    ax = axes[1]
    ax.scatter(*xpert[:, idx].T, c='tab:red', s=1**2, ec='none', alpha=0.5)
    ax.set_title('Perturbed (LPT1) positions', loc='left', fontsize=10)

    labels = ['x', 'y', 'z']
    for i, ax in enumerate(axes.flat):
        ax.set_aspect(1)
        # ax.set_xlim(0, Lbox[idx[0]])
        # ax.set_ylim(0, Lbox[idx[1]])
        ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
        ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    plt.show()

In [ ]:
if x.size <= 3*32**3:
    nr, nc = 1, 3
    fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

    labels = ['x', 'y', 'z']
    for i, ax in enumerate(axes.flat[:3]):
        ax.set_aspect(1)
        idx = [k for k in range(3) if k != i]
        disp = np.column_stack((x[:, idx].ravel(), xpert[:, idx].ravel()))
        ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
        # ax.set_xlim(-Lbox[idx[0]]/2, Lbox[idx[0]]/2)
        # ax.set_ylim(-Lbox[idx[1]]/2, Lbox[idx[1]]/2)
        ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
        ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    plt.show()

In [ ]:
def histogram(x, bins=50):
    '''TODO'''
    hist, edge = np.histogram(x, bins=bins)
    bin_c = (edge[:-1] + edge[1:]) / 2
    bin_w = np.diff(edge)  # Width of each bin
    return bin_c, hist, bin_w

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
c, h, w = histogram(np.linalg.norm((xpert - x)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.5)
ax.set_yscale('log')
ax.set_xlabel('Displacement [kpc/h]', fontsize=10)
ax.set_title(f'StepsIC 1-LPT displacements [z={z}]', loc='left', fontsize=10)

ax = next(axes)
c, h, w = histogram(np.linalg.norm(vpert/np.sqrt(a), axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.5)
ax.set_xlabel('Velocity [km/s]', fontsize=10)
ax.set_title(f'StepsIC 1-LPT velocities [z={z}]', loc='left', fontsize=10)

plt.show()

## 2LPT

In [ ]:
def lpt2(x, delta_k, nvox, dk, g2, aHf1, aHf2, h, counter=False):
    r'''
    Apply second-order Lagrangian Perturbation Theory (2LPT) to generate
    perturbed particle positions and velocities.

    This method provides a more accurate description of particle
    trajectories than the Zel'dovich approximation (1LPT) by including
    the second-order term in the displacement. The final (Eulerian)
    position :math:`\mathbf{x}` is computed from the initial (Lagrangian)
    position :math:`\mathbf{q}` as:

    .. math::
        \mathbf{x}(\mathbf{q}, t) = \mathbf{q} - D_1(t)\mathbf{\Psi}^{(1)}(\mathbf{q}) + D_2(t)\mathbf{\Psi}^{(2)}(\mathbf{q})

    where :math:`\mathbf{\Psi}^{(1)}` and :math:`\mathbf{\Psi}^{(2)}`
    are the first- and second-order displacement fields, and :math:`D_1`
    and :math:`D_2` are the corresponding linear and second-order growth
    factors.

    The **first-order field** is calculated from the overdensity :math:`\delta`
    as in 1LPT:
    
    .. math::
        \mathbf{\Psi}^{(1)}(\mathbf{k}) = -i \frac{\mathbf{k}}{|\mathbf{k}|^2} \delta(\mathbf{k}).

    The **second-order field** is derived from a scalar potential :math:`\phi^{(2)}`,
    where :math:`\mathbf{\Psi}^{(2)} = -\nabla\phi^{(2)}`. The potential
    itself is sourced by a quadratic source term :math:`S(\mathbf{x})`
    from the spatial derivatives of the first-order displacement.
    Following standard 2LPT theory, one may compute

    .. math::
        S(\mathbf{x}) =
            \frac{\partial \Psi^{(1)}_x}{\partial x}\,\frac{\partial \Psi^{(1)}_y}{\partial y}
            + \frac{\partial \Psi^{(1)}_x}{\partial x}\,\frac{\partial \Psi^{(1)}_z}{\partial z}
            + \frac{\partial \Psi^{(1)}_y}{\partial y}\,\frac{\partial \Psi^{(1)}_z}{\partial z}
            - \left[
                \left(\frac{\partial \Psi^{(1)}_x}{\partial y}\right)^2
                + \left(\frac{\partial \Psi^{(1)}_x}{\partial z}\right)^2
                + \left(\frac{\partial \Psi^{(1)}_y}{\partial z}\right)^2
            \right].
         
    In the code we denote:
        - $dPxx = \frac{\partial \Psi^{(1)}_x}{\partial x}$,
        - $dPxy = \frac{\partial \Psi^{(1)}_x}{\partial y}$,
        - $dPxz = \frac{\partial \Psi^{(1)}_x}{\partial z}$,
        - $dPyy = \frac{\partial \Psi^{(1)}_y}{\partial y}$,
        - $dPyz = \frac{\partial \Psi^{(1)}_y}{\partial z}$,
        - $dPzz = \frac{\partial \Psi^{(1)}_z}{\partial z}$,
         
    and then set

    .. math::
        S(\mathbf{x}) =
            dPxx\,dPyy + dPxx\,dPzz + dPyy\,dPzz - (dPxy^2 + dPxz^2 + dPyz^2).
        
    The Poisson equation in Fourier space is solved for the second-order
    potential:

    .. math::
        \phi^{(2)}(\mathbf{k}) = -\frac{S(\mathbf{k})}{|\mathbf{k}|^2},
         
    with the $k=0$ mode appropriately masked. The second-order displacement
    in Fourier space is then given by

    .. math::
        \Psi^{(2)}_i(\mathbf{k}) = i\,k_i\,\phi^{(2)}(\mathbf{k}),
         
    and an inverse FFT yields the real-space second-order displacement
    field.
    

    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Initial unperturbed particle positions (Lagrangian coordinates),
        in comoving Mpc/h.
    delta_k : ndarray
        A 3D complex-valued array of shape (Nx, Ny, Nz//2+1) representing
        the Fourier modes of the overdensity field.
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    aHf1 : float
        A prefactor for the velocity calculation, typically related to the
        time derivative of the growth factor (e.g. :math:`\dot{D}_1` or
        :math:`H(a)f(a)` where f is the growth rate). The code uses this
        in a non-standard velocity formula.
    g2 : float
        The second-order Lagrangian growth coefficient, typically
        :math:`g_2 = -\frac{3}{7} \Omega_M^{-1/143}`.
    aHf2 : float
        A prefactor for the second-order velocity term.
    h : float
        The dimensionless Hubble parameter, :math:`h = H_0 / 100`.
    counter : bool
        If True, applies a global sign flip to the Fourier-space density
        field (equivalent to a :math:`\pi` phase shift). This is useful
        for running "counter-phased" simulations to reduce sample
        variance.

    Returns
    -------
    xpert : ndarray of shape (N, 3)
        Perturbed particle positions (Eulerian coordinates) in comoving
        Mpc/h, wrapped within the periodic box.
    v : ndarray of shape (N, 3)
        Particle peculiar velocities in km/s, computed as:

        .. math::
            \mathbf{v} = \frac{-D_1 \cdot dD_1 \cdot \mathbf{\Psi}^{(1)} + D_2 \cdot dD_2 \cdot \mathbf{\Psi}^{(2)}}{h}.

    Notes
    -----
    **Overview of the Implementation:**

    1.  **First-Order Displacement:** The first-order displacement field,
        :math:`\mathbf{\Psi}^{(1)}`, is computed from the Fourier-space
        overdensity field, identical to the Zel'dovich approximation.

    2.  **Second-Order Source:** The spatial derivatives of the first-order
        field (i.e., the deformation tensor :math:`\partial \Psi_i^{(1)} / \partial q_j`)
        are calculated using FFTs. These are then combined to form the
        source term for the second-order potential.

    3.  **Second-Order Displacement:** The Poisson equation for the
        second-order potential :math:`\phi^{(2)}` is solved in Fourier
        space. The second-order displacement field, :math:`\mathbf{\Psi}^{(2)}`,
        is then found by taking the gradient of this potential, again in
        Fourier space.

    4.  **Interpolation and Particle Update:**
        - Both the first- and second-order displacement fields are
          interpolated from the grid to the Lagrangian particle positions.
        - Particle positions and velocities are updated by combining the
          interpolated first- and second-order contributions.
    '''
    kvec, kmod = fourier_grid(nvox, dk, hermitian=True)
    delta_k = delta_k * np.exp(1j * np.pi) if counter else delta_k
    mask = kmod > 0.0  # Avoid division by zero at k = 0

    # ------------------------------
    # 1. First-order displacement (Psi^(1))
    # ------------------------------
    phi1_k = np.zeros_like(kmod, dtype=complex)
    phi1_k[mask] = -delta_k[mask] / kmod[mask]**2  # Gravitational potential in Fourier space
    psi1_k = -1j * phi1_k[np.newaxis, ...] * kvec  # Displacement field in Fourier space
    disp_field1 = np.fft.irfftn(psi1_k, s=nvox, axes=(-3, -2, -1))

    # ------------------------------
    # 2. Compute derivatives of Psi^(1) for the second-order source
    # ------------------------------
    # The derivative of the i-th component of the first-order displacement
    # Psi^(1) with respect to the j-th coordinate in Fourier space is given by
    #
    #     d[Psi^(1)_i]/dx_j = irfftn(1j * psi1_k[i] * kvec[j])
    #
    axes = (0, 1, 2)
    dPxx = np.fft.irfftn(1j * psi1_k[0] * kvec[0], s=nvox, axes=axes)  # d(Psi_x)/dx
    dPxy = np.fft.irfftn(1j * psi1_k[0] * kvec[1], s=nvox, axes=axes)  # d(Psi_x)/dy
    dPxz = np.fft.irfftn(1j * psi1_k[0] * kvec[2], s=nvox, axes=axes)  # d(Psi_x)/dz
    # --
    dPyy = np.fft.irfftn(1j * psi1_k[1] * kvec[1], s=nvox, axes=axes)  # d(Psi_y)/dy
    dPyz = np.fft.irfftn(1j * psi1_k[1] * kvec[2], s=nvox, axes=axes)  # d(Psi_y)/dz
    # --
    dPzz = np.fft.irfftn(1j * psi1_k[2] * kvec[2], s=nvox, axes=axes)  # d(Psi_z)/dz

    # Compute the quadratic source S(x) and its Fourier transform S(k)
    S = dPxx * dPyy + dPxx * dPzz + dPyy * dPzz - (dPxy**2 + dPxz**2 + dPyz**2)
    S_k = np.fft.rfftn(S)

    # ------------------------------
    # 3. Second-order displacement (Psi^(2))
    # ------------------------------
    # Solve the Poisson equation in Fourier space, now for the source term S(k)
    #
    #     phi2(k) = -S(k) / |k|^2
    #
    phi2_k = np.zeros_like(S_k, dtype=complex)
    phi2_k[mask] = -S_k[mask] / kmod[mask]**2
    psi2_k = -1j * phi2_k[np.newaxis, ...] * kvec  # Displacement field in Fourier space
    disp_field2 = np.fft.irfftn(psi2_k, s=nvox, axes=(-3, -2, -1))

    # ------------------------------
    # 4. Interpolate and update particle positions and velocities
    # ------------------------------
    # For each spatial axis, interpolate the displacement fields (both
    # first- and second-order) from the grid to the particle positions.
    disp_field1_interp = np.empty_like(x, dtype=np.float32)
    disp_field2_interp = np.empty_like(x, dtype=np.float32)
    for i in range(3):
        disp_field1_interp[:, i] = interpolate_field(x, disp_field1[i], dk)
        disp_field2_interp[:, i] = interpolate_field(x, disp_field2[i], dk)
    
    xpert = x + disp_field1_interp + g2 * disp_field2_interp
    vpert = disp_field1_interp * aHf1 + g2 * disp_field2_interp * aHf2
    return xpert, vpert / h  # Mpc/h, km/s

In [ ]:
xpert, vpert = lpt2(x, delta_k, nvox, dk, g2, aHf1, aHf2, h=params['H'], counter=False)
log_lpt(x, xpert, vpert, title='2-LPT')
# xpert = np.where(periodic, np.mod(xpert, Lbox), xpert)

INFO:__main__:2-LPT 'x' displacements:	d_max = 508.694 kpc/h; d_avg = 87.652 kpc/h
INFO:__main__:2-LPT 'vx' velocities:	v_max = 226.885 km/s; v_avg = 39.035 km/s
INFO:__main__:2-LPT 'y' displacements:	d_max = 468.036 kpc/h; d_avg = 86.408 kpc/h
INFO:__main__:2-LPT 'vy' velocities:	v_max = 208.473 km/s; v_avg = 38.481 km/s
INFO:__main__:2-LPT 'z' displacements:	d_max = 493.762 kpc/h; d_avg = 89.427 kpc/h
INFO:__main__:2-LPT 'vz' velocities:	v_max = 220.159 km/s; v_avg = 39.825 km/s

In [ ]:
if x.size <= 3*36**3:
    nr, nc = 1, 2
    fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

    idx = [0, 1]

    ax = axes[0]
    ax.scatter(*x[:, idx].T, c='k', s=1**2, ec='none', alpha=0.5)
    ax.set_title('Initial positions', loc='left', fontsize=10)

    ax = axes[1]
    ax.scatter(*xpert[:, idx].T, c='tab:blue', s=1**2, ec='none', alpha=0.5)
    ax.set_title('Perturbed (LPT2) positions', loc='left', fontsize=10)

    labels = ['x', 'y', 'z']
    for i, ax in enumerate(axes.flat):
        ax.set_aspect(1)
        ax.set_xlim(-Lbox[idx[0]]/2, Lbox[idx[0]]/2)
        ax.set_ylim(-Lbox[idx[1]]/2, Lbox[idx[1]]/2)
        ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
        ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    plt.show()

In [ ]:
if x.size <= 3*32**3:
    nr, nc = 1, 3
    fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

    labels = ['x', 'y', 'z']
    for i, ax in enumerate(axes.flat[:3]):
        ax.set_aspect(1)
        idx = [k for k in range(3) if k != i]
        disp = np.column_stack((x[:, idx].ravel(), xpert[:, idx].ravel()))
        ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
        ax.set_xlim(-Lbox[idx[0]]/2, Lbox[idx[0]]/2)
        ax.set_ylim(-Lbox[idx[1]]/2, Lbox[idx[1]]/2)
        ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
        ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    plt.show()

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
c, h, w = histogram(np.linalg.norm((xpert - x)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.5)
ax.set_yscale('log')
ax.set_xlabel('Displacement [kpc/h]', fontsize=10)
ax.set_title(f'StepsIC 2-LPT displacements [z={z}]', loc='left', fontsize=10)

ax = next(axes)
c, h, w = histogram(np.linalg.norm(vpert/np.sqrt(a), axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.5)
ax.set_xlabel('Velocity [km/s]', fontsize=10)
ax.set_title(f'StepsIC 2-LPT velocities [z={z}]', loc='left', fontsize=10)

plt.show()

## Comparison of models

In [ ]:
xpert_lpt1, vpert_lpt1 = lpt1(x, delta_k, nvox, dk, aHf1, h=params['H'], counter=False)
log_lpt(x, xpert_lpt1, vpert_lpt1, title='1-LPT')
print()
xpert_lpt2, vpert_lpt2 = lpt2(x, delta_k, nvox, dk, g2, aHf1, aHf2, h=params['H'], counter=False)
log_lpt(x, xpert_lpt2, vpert_lpt2, title='2-LPT')

In [ ]:
if x.size <= 3*36**3:
    nr, nc = 1, 3
    fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

    labels = ['x', 'y', 'z']
    for i, ax in enumerate(axes.flat[:3]):
        ax.set_aspect(1)
        idx = [k for k in range(3) if k != i]
        ax.scatter(*x[:, idx].T, c='k', s=2**2, ec='none', alpha=0.5)
        ax.scatter(*xpert_lpt1[:, idx].T, c='tab:red', s=1**2, ec='none', alpha=0.5)
        ax.scatter(*xpert_lpt2[:, idx].T, c='tab:blue', s=1**2, ec='none', alpha=0.5)
        lidx = [labels[idx[0]], labels[idx[1]]]
        ax.set_xlabel(f'{lidx[0]} [Mpc/h]')
        ax.set_ylabel(f'{lidx[1]} [Mpc/h]')
        ax.set_title(f'1-LPT and 2-LPT perturbations [{lidx[0]}, {lidx[1]}]',
                    loc='left', fontsize=10)
        handles = [
            Line2D([0], [0], color='tab:red', lw=0, marker='o', ms=4, label='1-LPT'),
            Line2D([0], [0], color='tab:blue', lw=0, marker='o', ms=4, label='2-LPT')
        ]
        ax.legend(handles=handles, loc='upper right', fontsize=8)
    plt.show()

In [ ]:
if x.size <= 3*24**3:
    nr, nc = 1, 3
    fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

    labels = ['x', 'y', 'z']
    for i, ax in enumerate(axes.flat[:3]):
        ax.set_aspect(1)
        idx = [k for k in range(3) if k != i]
        disp = np.column_stack((x[:, idx].ravel(), xpert_lpt1[:, idx].ravel()))
        ax.plot(*disp, lw=1, c='tab:red', alpha=0.5)
        disp = np.column_stack((x[:, idx].ravel(), xpert_lpt2[:, idx].ravel()))
        ax.plot(*disp, lw=1, c='tab:blue', alpha=0.5)
        ax.set_xlim(0, Lbox[idx[0]])
        ax.set_ylim(0, Lbox[idx[1]])
        lidx = [labels[idx[0]], labels[idx[1]]]
        ax.set_xlabel(f'{lidx[0]} [Mpc/h]')
        ax.set_ylabel(f'{lidx[1]} [Mpc/h]')
        ax.set_title(f'1-LPT and 2-LPT perturbations [{lidx[0]}, {lidx[1]}]',
                    loc='left', fontsize=10)
        handles = [
            Line2D([0], [0], color='tab:red', lw=2, label='1-LPT'),
            Line2D([0], [0], color='tab:blue', lw=2, label='2-LPT')
        ]
        ax.legend(handles=handles, loc='upper right', fontsize=8)
    plt.show()

In [ ]:
if x.size <= 3*24**3:
    nr, nc = 1, 3
    fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

    labels = ['x', 'y', 'z']
    for i, ax in enumerate(axes.flat[:3]):
        ax.set_aspect(1)
        idx = [k for k in range(3) if k != i]
        disp = np.column_stack((xpert_lpt1[:, idx].ravel(), xpert_lpt2[:, idx].ravel()))
        ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
        ax.set_xlim(0, Lbox[idx[0]])
        ax.set_ylim(0, Lbox[idx[1]])
        ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
        ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
        ax.set_title('Difference between LPT1 and LPT2', loc='left', fontsize=10)
    plt.show()

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

handles = [
    Line2D([0], [0], color='tab:red', lw=5, label='StepsIC 1-LPT'),
    Line2D([0], [0], color='tab:blue', lw=5, label='StepsIC 2-LPT')
]

ax = next(axes)
c, h, w = histogram(np.linalg.norm((xpert_lpt1 - x)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.4)
c, h, w = histogram(np.linalg.norm((xpert_lpt2 - x)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.4)
ax.set_yscale('log')
ax.set_xlabel('Displacement [kpc/h]', fontsize=10)
ax.set_title('StepsIC displacements', loc='left', fontsize=10)
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
c, h, w = histogram(np.linalg.norm(vpert_lpt1, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.4)
c, h, w = histogram(np.linalg.norm(vpert_lpt2, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.4)
ax.set_xlabel('Velocity [km/s]', fontsize=10)
ax.set_title('StepsIC velocities', loc='left', fontsize=10)
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/xv_grid_N{nmesh}_L{np.max(Lbox)}_steps1-2.png', bbox_inches='tight', dpi=300)
plt.show()

### Compare with `monofonic`

#### 1-LPT

In [ ]:
# file_path = 'output/ics_lpt1.hdf5'
# with h5py.File(file_path, 'r') as f:
#     cx = np.array(f['DM_dx'])
#     cy = np.array(f['DM_dy'])
#     cz = np.array(f['DM_dz'])
#     xpert_m1 = np.column_stack((cx, cy, cz)) + dk/2  # TODO: fix -> (3, nx, ny, nz)
#     vx = np.array(f['DM_vx'])
#     vy = np.array(f['DM_vy'])
#     vz = np.array(f['DM_vz'])
#     vpert_m1 = np.column_stack((vx, vy, vz))  # TODO: fix -> (3, nx, ny, nz)

In [ ]:
# Load HDF5 file from a folder in a Gadget format
file_path = 'output/ics_gadget_N128_L1000_z63_lpt1.hdf5'
with h5py.File(file_path, 'r') as f:
    a_m1 = f['Header'].attrs['Time'][0]
    xpert_m1 = np.array(f['PartType1']['Coordinates']) + dk/2
    vpert_m1 = np.array(f['PartType1']['Velocities']) * np.sqrt(a)  # Gadget

cm, hm, wm = histogram(np.linalg.norm(vpert_m1, axis=1), bins=100)
cs, hs, ws = histogram(np.linalg.norm(vpert_lpt1, axis=1), bins=100)
print(np.mean(cm/cs))
log_lpt(x, xpert_lpt1, vpert_lpt1, title='1-LPT')
print()
log_lpt(x, xpert_m1 - 500, vpert_m1, title='1-LPT')

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

handles = [
    Line2D([0], [0], color='0.6', lw=5, label='MonofonIC'),
    Line2D([0], [0], color='tab:red', lw=5, label='StepsIC 1-LPT')
]

ax = next(axes)
c, h, w = histogram(np.linalg.norm((xpert_m1 - x - 500)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='0.6')
c, h, w = histogram(np.linalg.norm((xpert_lpt1 - x)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.4)
ax.set_yscale('log')
ax.set_xlabel('Displacement [kpc/h]', fontsize=10)
ax.set_title(f'StepsIC 1-LPT displacements [z={z}]', loc='left', fontsize=10)
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
c, h, w = histogram(np.linalg.norm(vpert_m1, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='0.6')
c, h, w = histogram(np.linalg.norm(vpert_lpt1, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.4)
ax.set_xlabel('Velocity [km/s]', fontsize=10)
ax.set_title(f'StepsIC 1-LPT velocities [z={z}]', loc='left', fontsize=10)
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/xv_grid_N{nmesh}_L{np.max(Lbox)}_steps1-mf.png', bbox_inches='tight', dpi=300)
plt.show()

#### 2-LPT

In [ ]:
# Load HDF5 file from a folder
file_path = 'output/ics_gadget_N128_L1000_z63_lpt2.hdf5'
with h5py.File(file_path, 'r') as f:
    xpert_m2 = np.array(f['PartType1']['Coordinates']) + dk/2
    vpert_m2 = np.array(f['PartType1']['Velocities']) * np.sqrt(a)  # Gadget

# cm, hm, wm = histogram(np.linalg.norm(vpert_m2, axis=1), bins=100)
# cs, hs, ws = histogram(np.linalg.norm(vpert_lpt2, axis=1), bins=100)
# np.mean(cm/cs)
log_lpt(x, xpert_lpt2, vpert_lpt2, title='2-LPT')
print()
log_lpt(x+500, xpert_m2, vpert_m2, title='2-LPT')

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

handles = [
    Line2D([0], [0], color='0.6', lw=5, label='MonofonIC'),
    Line2D([0], [0], color='tab:blue', lw=5, label='StepsIC 2-LPT')
]

ax = next(axes)
c, h, w = histogram(np.linalg.norm((xpert_m2 - x - 500)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='0.6')
c, h, w = histogram(np.linalg.norm((xpert_lpt2 - x)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.4)
ax.set_yscale('log')
ax.set_xlabel('Displacement [kpc/h]', fontsize=10)
ax.set_title(f'StepsIC 2-LPT displacements [z={z}]', loc='left', fontsize=10)
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
c, h, w = histogram(np.linalg.norm(vpert_m2, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='0.6')
c, h, w = histogram(np.linalg.norm(vpert_lpt2, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.4)
ax.set_yscale('log')
ax.set_xlabel('Velocity [km/s]', fontsize=10)
ax.set_title(f'StepsIC 2-LPT velocities [z={z}]', loc='left', fontsize=10)
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/xv_grid_N{nmesh}_L{np.max(Lbox)}_steps2-mf.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt1 = pk_particle(xpert_lpt1, nvox, Lbox)
ax.loglog(pk_pt1['k'], pk_pt1['power'].real,
          color='tab:red', lw=1.5, label='Particle P(k)')
pk_pt1 = pk_particle(xpert_m1, nvox, Lbox)
ax.loglog(pk_pt1['k'], pk_pt1['power'].real,
          color='tab:green', lw=1.5, ls='--', label='MonofonIC P(k)')
ax.set_xlim(None, 2.0)
ax.set_ylim(1e-3, None)
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('MonofonIC vs StePS (1-LPT))', loc='left', fontsize=10)
ax.legend(loc='lower left', fontsize=10, frameon=False)

ax = next(axes)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt2 = pk_particle(xpert_lpt2, nvox, Lbox)
ax.loglog(pk_pt2['k'], pk_pt2['power'].real,
          color='tab:blue', lw=1.5, label='Particle P(k)')
pk_pt2 = pk_particle(xpert_m2, nvox, Lbox)
ax.loglog(pk_pt2['k'], pk_pt2['power'].real,
          color='tab:orange', lw=1.5, ls='--', label='MonofonIC P(k)')
ax.set_xlim(None, 2.0)
ax.set_ylim(1e-3, None)
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('MonofonIC vs StePS (2-LPT)', loc='left', fontsize=10)
ax.legend(loc='lower left', fontsize=10, frameon=False)

fig.savefig(f'output/pk_grid_N{nmesh}_L{np.max(Lbox)}_compare.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

handles = [
    Line2D([0], [0], color='tab:red', lw=5, label='1-LPT'),
    Line2D([0], [0], color='tab:blue', lw=5, label='2-LPT')
]

ax = next(axes)
c, h, w = histogram(np.linalg.norm((xpert_m1 - x - 500)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.4)
c, h, w = histogram(np.linalg.norm((xpert_m2 - x - 500)*1000, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.4)
ax.set_yscale('log')
ax.set_xlabel('Displacement [kpc/h]', fontsize=10)
ax.set_title('MonofonIC displacements', loc='left', fontsize=10)
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
c, h, w = histogram(np.linalg.norm(vpert_m1, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.4)
c, h, w = histogram(np.linalg.norm(vpert_m2, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.4)
ax.set_xlabel('Velocity [km/s]', fontsize=10)
ax.set_title('MonofonIC velocities', loc='left', fontsize=10)
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/xv_grid_N{nmesh}_L{np.max(Lbox)}_mf1-2.png', bbox_inches='tight', dpi=300)
plt.show()

## Compact funtion

In [ ]:
z = 63
a = 1.0 / (1.0 + z)
de_model = 'Lambda'
de_params = []  # No parameters needed for LCDM model

D1 = D1_z(
    z, H0=params['H0'], omega_m=params['omega_m'], omega_b=params['omega_b'],
    omega_l=params['omega_l'], ns=params['ns'], sigma8=params['sigma8'],
    de_model=de_model, de_params=de_params)
D2 = (3.0/7.0) * D1**2 * params['omega_m']**(-1/143)  # Bernardeau et al. 2001, eq. 97
print(f'D1(z={z}) = {D1:.6f}, D2(z={z}) = {D2:.6f}')

# Bernardeau et al. 2001, eq. 99
aHf1 = a * hubble_a(a, params['H0'], params['omega_m'], params['omega_l'])
aHf1 *= F_omega(a, params['omega_m'], params['omega_l'])
aHf2 = a * hubble_a(a, params['H0'], params['omega_m'], params['omega_l'])
aHf2 *= F2_omega(a, params['omega_m'], params['omega_l'])
print(f'1st vel. prefac(z={z}) = {aHf1:.6f}, 2nd vel. prefac(z={z}) = {aHf2:.6f}')

In [ ]:
def generate_ic(params, nmesh, Lbox, periodic, kmax=100.0, npoints=2048, seed=None):
    nvox, dk = cubic_voxels(nmesh, Lbox)

    kmin = np.pi / np.min(Lbox)
    camb_params = init_camb_cosmology(
        H0=params['H0'], ombh2=params['ombh2'], omch2=params['omch2'],
        omk=params.get('omk', 0.0), mnu=params['mnu'], nnu=params['nnu'],
        YHe=params['YHe'], TCMB=params['Tcmb'], zrei=params['zrei'],
        de_model=de_model, de_params=de_params, nonlinear=False)
    kh, pk = camb_spectrum(
            camb_params, z=0, As=params['As'], ns=params['ns'],
            sigma8_init=params['sigma8'], kmin=kmin, kmax=kmax, npoints=npoints)
    pk = pk[0]*D1**2  # Scale P(k) by D1^2 for z=0

    field = white_noise(nvox=nvox, seed=seed)
    delta_k = generate_delta_k(kh, pk, nvox, dk, Lbox, field=field)

    x = create_grid(nvox, dk)
    xpert, vpert = lpt1(x, delta_k, nvox, dk, D1, aHf1, h=params['h'], counter=False)
    xpert = np.where(periodic, np.mod(xpert, Lbox), xpert)

    return x, xpert, vpert